# Discovery + Silver: `university.students`

Primer ejemplo del patron que vamos a repetir para las 18 tablas: leer la tabla cruda de `bronze` con pandas, perfilarla (nulos, duplicados, tipos, rangos raros), limpiarla, y escribir el resultado en `silver` -- todo dentro de esta misma notebook.

Flujo del proyecto: **bronze** (crudo tal cual llega) -> **silver** (limpio, tipado -- lo que hacemos aca, con pandas) -> **gold** (modelo estrella con SQL, dims + facts, para sacar informacion de negocio).

In [1]:
import sys
from pathlib import Path
sys.path.append("/home/jovyan/work/src")

import pandas as pd
from utils.db import get_engine, get_psycopg2_connection

engine = get_engine()
SQL_SILVER = Path("/home/jovyan/work/sql/silver")

def run_sql_file(path):
    sql = path.read_text()
    conn = get_psycopg2_connection()
    try:
        with conn.cursor() as cur:
            cur.execute(sql)
        conn.commit()
        print(f"OK: {path.name} ejecutado")
    finally:
        conn.close()

df = pd.read_sql("SELECT * FROM bronze.university__students", engine)
df.shape

(5000, 10)

## 1. Forma general

Todo llega como `TEXT` desde bronze (a proposito, ver `docs/decisiones.md`). Miramos columnas, tipos actuales y una muestra.

In [2]:
print(df.dtypes)
df.head()

student_id              object
first_name              object
last_name               object
email                   object
birth_date              object
enrolled_at             object
country                 object
_source_file            object
_ingested_at    datetime64[ns]
_dag_run_id             object
dtype: object


,student_id,first_name,last_name,email,birth_date,enrolled_at,country,_source_file,_ingested_at,_dag_run_id
0,STU-0000001,Martina,Diaz,martina.diaz5727@lake.local,2000-12-21,2019-10-01,US,university/students.csv,2026-07-21 09:50:01.912437,manual__2026-07-21T09:49:58+00:00
1,STU-0000002,Manuel,Torres,manuel.torres5619@mail.test,2004-08-10,2025-03-20,CL,university/students.csv,2026-07-21 09:50:01.912437,manual__2026-07-21T09:49:58+00:00
2,STU-0000003,Maximiliano,Martinez,maximiliano.martinez7688@demo.io,2007-08-10,2022-10-19,PE,university/students.csv,2026-07-21 09:50:01.912437,manual__2026-07-21T09:49:58+00:00
3,STU-0000004,Magdalena,Vasquez,magdalena.vasquez8686@example.com,2002-05-13,2020-09-23,CL,university/students.csv,2026-07-21 09:50:01.912437,manual__2026-07-21T09:49:58+00:00
4,STU-0000005,Luis,Rivera,luis.rivera9349@lake.local,2005-12-18,2025-07-28,CL,university/students.csv,2026-07-21 09:50:01.912437,manual__2026-07-21T09:49:58+00:00


## 2. Nulos y duplicados

In [3]:
print("Nulos por columna:")
print(df.isna().sum())
print()
print("student_id duplicados:", df["student_id"].duplicated().sum())
print("Filas 100% duplicadas:", df.duplicated().sum())

Nulos por columna:
student_id      0
first_name      0
last_name       0
email           0
birth_date      0
enrolled_at     0
country         0
_source_file    0
_ingested_at    0
_dag_run_id     0
dtype: int64

student_id duplicados: 0
Filas 100% duplicadas: 0


## 3. Rangos y valores raros

- `country`: cuantos valores distintos hay, y si alguno se ve raro (typo, casing inconsistente).
- Fechas: que `birth_date` y `enrolled_at` sean fechas validas, y que `birth_date` sea siempre anterior a `enrolled_at` (nadie se inscribe antes de nacer).

In [4]:
print("Valores distintos de country:")
print(df["country"].value_counts())
print()

birth = pd.to_datetime(df["birth_date"])
enrolled = pd.to_datetime(df["enrolled_at"])

print("Fechas invalidas (no parsean):", birth.isna().sum() + enrolled.isna().sum())
print("Filas con birth_date >= enrolled_at (inconsistente):", (birth >= enrolled).sum())

Valores distintos de country:
country
CL    1980
AR     523
PE     513
MX     483
BR     427
ES     390
CO     389
US     295
Name: count, dtype: int64

Fechas invalidas (no parsean): 0
Filas con birth_date >= enrolled_at (inconsistente): 0


## 4. Conclusion: reglas de limpieza

`students` resulta ser una tabla limpia (sin nulos, sin duplicados, sin fechas invertidas, `country` con 8 codigos ISO consistentes). No hace falta descartar ni corregir filas -- las reglas son solo de **tipado y estandarizacion**:

- `first_name`, `last_name` -> `strip()`.
- `email` -> `strip()` + minusculas (estandarizar para poder hacer joins/comparaciones confiables despues).
- `birth_date`, `enrolled_at` -> castear de texto a fecha real.
- `country` -> `strip()` + mayusculas (ya vienen consistentes, se fuerza el estandar de todas formas).
- `student_id` ya es unico, queda como esta (va a ser la primary key en silver).

## 5. Limpieza con pandas

In [5]:
df_silver = df[["student_id", "first_name", "last_name", "email", "birth_date", "enrolled_at", "country"]].copy()

df_silver["first_name"] = df_silver["first_name"].str.strip()
df_silver["last_name"] = df_silver["last_name"].str.strip()
df_silver["email"] = df_silver["email"].str.strip().str.lower()
df_silver["birth_date"] = pd.to_datetime(df_silver["birth_date"]).dt.date
df_silver["enrolled_at"] = pd.to_datetime(df_silver["enrolled_at"]).dt.date
df_silver["country"] = df_silver["country"].str.strip().str.upper()

df_silver.head()

,student_id,first_name,last_name,email,birth_date,enrolled_at,country
0,STU-0000001,Martina,Diaz,martina.diaz5727@lake.local,2000-12-21,2019-10-01,US
1,STU-0000002,Manuel,Torres,manuel.torres5619@mail.test,2004-08-10,2025-03-20,CL
2,STU-0000003,Maximiliano,Martinez,maximiliano.martinez7688@demo.io,2007-08-10,2022-10-19,PE
3,STU-0000004,Magdalena,Vasquez,magdalena.vasquez8686@example.com,2002-05-13,2020-09-23,CL
4,STU-0000005,Luis,Rivera,luis.rivera9349@lake.local,2005-12-18,2025-07-28,CL


## 6. Validar antes de escribir

Chequeo rapido de que la limpieza no rompio nada (mismo numero de filas, sin nulos nuevos, PK sigue unica) antes de escribir en `silver`.

In [6]:
assert len(df_silver) == len(df), "se perdieron o duplicaron filas en la limpieza"
assert df_silver.isna().sum().sum() == 0, "aparecieron nulos nuevos"
assert df_silver["student_id"].is_unique, "student_id ya no es unico"
print("OK:", len(df_silver), "filas listas para silver")

OK: 5000 filas listas para silver


## 7. Escribir en `silver.university__students`

Ahora con esquema explicito (`sql/silver/university.sql`: `PRIMARY KEY`, `NOT NULL`) en vez de dejar que `pandas.to_sql()` infiera tipos sin restricciones -- ver `docs/decisiones.md` #22. El notebook sigue siendo el ETL completo: crea la tabla (si no existe), la trunca, e inserta el dataframe ya limpio -- mismo criterio de idempotencia que antes (full-refresh, sin merge/upsert porque el CSV fuente es estatico).

In [7]:
df_silver["_silver_loaded_at"] = pd.Timestamp.utcnow()

run_sql_file(SQL_SILVER / "university.sql")

conn = get_psycopg2_connection()
with conn.cursor() as cur:
    cur.execute("TRUNCATE TABLE silver.university__students CASCADE;")
conn.commit()
conn.close()

df_silver.to_sql(
    "university__students",
    engine,
    schema="silver",
    if_exists="append",
    index=False,
    method="multi",
    chunksize=1000,
)
print("Escrito en silver.university__students")

OK: university.sql ejecutado


Escrito en silver.university__students


## 8. Verificar lo que quedo en Postgres

In [8]:
check = pd.read_sql("SELECT * FROM silver.university__students LIMIT 5", engine)
print(pd.read_sql("SELECT count(*) AS filas, count(DISTINCT student_id) AS ids_unicos FROM silver.university__students", engine))
check

   filas  ids_unicos
0   5000        5000


,student_id,first_name,last_name,email,birth_date,enrolled_at,country,_silver_loaded_at
0,STU-0000001,Martina,Diaz,martina.diaz5727@lake.local,2000-12-21,2019-10-01,US,2026-07-21 09:50:19.163862+00:00
1,STU-0000002,Manuel,Torres,manuel.torres5619@mail.test,2004-08-10,2025-03-20,CL,2026-07-21 09:50:19.163862+00:00
2,STU-0000003,Maximiliano,Martinez,maximiliano.martinez7688@demo.io,2007-08-10,2022-10-19,PE,2026-07-21 09:50:19.163862+00:00
3,STU-0000004,Magdalena,Vasquez,magdalena.vasquez8686@example.com,2002-05-13,2020-09-23,CL,2026-07-21 09:50:19.163862+00:00
4,STU-0000005,Luis,Rivera,luis.rivera9349@lake.local,2005-12-18,2025-07-28,CL,2026-07-21 09:50:19.163862+00:00
